# 店舗売上需要予測: 実験 #015 メダル圏突破アルティメット・ブレンド

- **コンペ名:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **目的:** 
  1. ローカル最高峰モデル群と、Google Colab (T4 GPU) で獲得した 1,000 木深層モデルを対数空間で完全統合
  2. **Exp #012 (現自己ベスト: 0.38908 / 71位):** 重み 40%
  3. **Exp #014 (Colab T4 GPU 1,000木: 0.39019):** 重み 25%
  4. **Exp #013 (曜日グループ平均単体: 0.38972):** 重み 20%
  5. **Exp #010 (特売平常比・本番最強: 0.38968):** 重み 15%
  6. 直近21日完全ゼロ売上実績の 124 系列に対するゼロクリップ後処理の厳格適用
- **評価指標:** **RMSLE (Root Mean Squared Logarithmic Error)**
- **結果:** **Public LB `0.38856` (69位 / 542チーム中 念願の60位台突入！自己ベスト更新！🏆)**
  - Top 10%（メダル圏、Rank 54: `0.38602`）まで残り **0.00254** / 15人抜き圏内へ到達！

## 1. 予測ファイルの読み込みと相関行列の確認

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

base_dir = '../'
sub12 = pd.read_csv(os.path.join(base_dir, 'submission_exp012.csv'))
sub13 = pd.read_csv(os.path.join(base_dir, 'submission_exp013.csv'))
sub14 = pd.read_csv(os.path.join(base_dir, 'submission_colab_gpu.csv'))
sub10 = pd.read_csv(os.path.join(base_dir, 'submission_exp010.csv'))

pred_df = pd.DataFrame({
    'Exp012 (0.38908)': sub12['sales'],
    'Exp014 (0.39019)': sub14['sales'],
    'Exp013 (0.38972)': sub13['sales'],
    'Exp010 (0.38968)': sub10['sales']
})
print('--- 各モデルの平均予測売上 ---')
print(pred_df.mean())
print('\n--- 予測相関行列 ---')
print(pred_df.corr())

## 2. 対数空間加重ブレンド (Log-space Ultimate Blending)

$$\log(1 + \hat{y}_{\text{blend}}) = 0.40 \log(1 + y_{012}) + 0.25 \log(1 + y_{014}) + 0.20 \log(1 + y_{013}) + 0.15 \log(1 + y_{010})$$

In [ ]:
w12, w14, w13, w10 = 0.40, 0.25, 0.20, 0.15

log12 = np.log1p(np.clip(sub12['sales'].values, 0, None))
log14 = np.log1p(np.clip(sub14['sales'].values, 0, None))
log13 = np.log1p(np.clip(sub13['sales'].values, 0, None))
log10 = np.log1p(np.clip(sub10['sales'].values, 0, None))

log_blend = w12 * log12 + w14 * log14 + w13 * log13 + w10 * log10
sales_blend = np.clip(np.expm1(log_blend), 0, None)
print(f'対数ブレンド後 平均売上: {np.mean(sales_blend):.2f}')

## 3. 直近21日ゼロ売上フィルター（124系列）の厳格適用

In [ ]:
train_df = pd.read_csv(os.path.join(base_dir, 'data', 'train.csv'))
test_df = pd.read_csv(os.path.join(base_dir, 'data', 'test.csv'))
train_df['date'] = pd.to_datetime(train_df['date'])

recent_train = train_df[train_df['date'] >= '2017-07-25']
recent_sum = recent_train.groupby(['store_nbr', 'family'])['sales'].sum().reset_index()
zero_pairs = set(zip(recent_sum[recent_sum['sales'] == 0]['store_nbr'], recent_sum[recent_sum['sales'] == 0]['family']))
print(f'直近21日完全ゼロ系列数: {len(zero_pairs)} 系列 (1,984行)')

sub_df = pd.DataFrame({'id': sub12['id'], 'sales': sales_blend})
test_merged = pd.merge(test_df[['id', 'store_nbr', 'family']], sub_df, on='id')
zero_mask = [((st, fam) in zero_pairs) for st, fam in zip(test_merged['store_nbr'], test_merged['family'])]
sales_blend[zero_mask] = 0.0

print(f'クリップ後ゼロ予測件数: {(sales_blend == 0).sum()} 件')
print(f'最終平均売上: {np.mean(sales_blend):.2f}')

## 4. 提出ファイルの生成と確認

In [ ]:
final_sub = pd.DataFrame({
    'id': sub12['id'],
    'sales': sales_blend
}).sort_values('id').reset_index(drop=True)

out_path = os.path.join(base_dir, 'submission_exp015.csv')
final_sub.to_csv(out_path, index=False)
print(f'提出ファイル保存完了: {out_path} ({len(final_sub)} 行)')
print(final_sub.head(10))